# Object Detection Tutorial: YOLO11n on COCO128

Welcome! In this tutorial you will learn the core ideas behind **object detection** by using a real, pretrained model (**YOLO11n**) on a real dataset (**COCO128**).

This notebook is designed to run in **Google Colab**. Before running any cells:

1. Go to `Runtime > Change runtime type`
2. Set **Hardware accelerator** to **GPU** (T4 is fine)
3. Run the cells from top to bottom

We will cover:
1. Understanding the problem (classification vs. detection)
2. Loading and inspecting the dataset
3. Running detection and understanding predictions (confidence, IoU, NMS)
4. Training and evaluating YOLO11n
5. Evaluation metrics (precision, recall, F1, confusion matrix)
6. Experimenting with key parameters (confidence threshold, NMS IoU threshold)

No prior object detection experience is assumed — every metric is explained with a simple, concrete example before we use it in code.


In [ ]:
# Install the Ultralytics package (includes YOLO11 and dataset download utilities)
!pip install -q ultralytics

import ultralytics
ultralytics.checks()  # prints Python, PyTorch, GPU/CPU info

## 1 — Understand the problem

### 1.1 Learning goals

By the end of this notebook you will be able to:

- Explain the difference between image classification and object detection
- Load a pretrained detection model and understand the YOLO dataset format
- Run inference and interpret the raw prediction output (boxes, classes, confidence scores)
- Define and compute **IoU**, understand **confidence score** and **Non-Maximum Suppression (NMS)**
- Train YOLO11n on a small dataset and read the training logs/graphs
- Evaluate a trained model using **precision**, **recall**, **F1 score**, and the **confusion matrix**
- Understand how changing the **confidence threshold** and **NMS IoU threshold** affects predictions

### 1.2 Image classification vs. object detection

| | Image Classification | Object Detection |
|---|---|---|
| **Question it answers** | "What is in this image?" | "What objects are in this image, and *where*?" |
| **Output** | A single label (+ confidence) for the whole image | A list of boxes, each with a label and confidence |
| **Example output** | `"cat"` | `"cat" at box (x1=34, y1=12, x2=210, y2=190)` |
| **Handles multiple objects?** | No — one label per image | Yes — one box per detected object |

**Conceptual example:** Given a photo of a street with two cars and one pedestrian:
- A **classifier** might output: `"street scene"` (one label for the whole image).
- A **detector** outputs three separate boxes: `car`, `car`, `person`, each with its own location and confidence score.

Object detection is strictly harder because the model must both **localize** (find *where*) and **classify** (find *what*) every object in the image.

## 2 — Load and inspect the dataset

### 2.1 Load a pretrained model

YOLO11 detection checkpoints (like `yolo11n.pt`, the "n" stands for **nano**, the smallest/fastest version) are **pretrained on the COCO dataset**, which has 80 common object classes (person, car, dog, chair, ...).

Loading a pretrained checkpoint means the model already knows how to detect these 80 classes out of the box — no training required to try it out.

In [ ]:
from ultralytics import YOLO

# Load the YOLO11n checkpoint pretrained on COCO (downloads automatically the first time)
model = YOLO("yolo11n.pt")

# The 80 COCO class names the model already knows how to detect
print(f"Number of classes: {len(model.names)}")
print(model.names)

### 2.2 Understand the YOLO dataset structure

We will use **COCO128** — a tiny 128-image subset of COCO, perfect for a quick, beginner-friendly tutorial (it trains in minutes instead of hours).

Downloading it (via `check_det_dataset`) reveals the standard YOLO dataset layout:

```
coco128/
├── images/
│   └── train2017/   <- the .jpg image files
└── labels/
    └── train2017/   <- one .txt label file per image (same filename)
```

Each label `.txt` file has **one line per object**, in the format:

```
class_id  x_center  y_center  width  height
```

- `class_id`: integer index into the list of class names (e.g. `0` = person)
- `x_center, y_center, width, height`: all **normalized to [0, 1]** relative to the image width/height (so the format works for any image size)

Let's download the dataset and look at a real label file.

In [ ]:
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

# Downloads (if needed) and validates the COCO128 dataset, returns info about it
data_info = check_det_dataset("coco128.yaml")

dataset_root = Path(data_info["path"])
image_dir = dataset_root / "images" / "train2017"
label_dir = dataset_root / "labels" / "train2017"

image_files = sorted(image_dir.glob("*.jpg"))
label_files = sorted(label_dir.glob("*.txt"))

print(f"Dataset root: {dataset_root}")
print(f"Number of images: {len(image_files)}")
print(f"Number of label files: {len(label_files)}")

# Look at the contents of one label file
sample_label = label_files[0]
print(f"\nContents of {sample_label.name}:")
print(sample_label.read_text())

### 2.3 Visualize ground-truth labels

Numbers in a `.txt` file are hard to interpret by eye. Let's write a small helper that draws the ground-truth boxes on top of the image, so we can *see* what the labels represent.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

class_names = data_info["names"]  # dict: class_id -> class name


def plot_yolo_labels(image_path, label_path, class_names, ax=None):
    """Draw YOLO-format ground-truth boxes on an image."""
    image = Image.open(image_path)
    img_w, img_h = image.size

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image)

    for line in Path(label_path).read_text().splitlines():
        class_id, xc, yc, w, h = map(float, line.split())
        # Convert normalized center/width/height -> pixel corner coordinates
        x1 = (xc - w / 2) * img_w
        y1 = (yc - h / 2) * img_h
        box_w = w * img_w
        box_h = h * img_h

        rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, class_names[int(class_id)], color="lime", fontsize=10, weight="bold")

    ax.set_title(Path(image_path).name)
    ax.axis("off")


# Visualize a few sample images with their ground-truth labels
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ax in enumerate(axes):
    plot_yolo_labels(image_files[i], label_files[i], class_names, ax=ax)
plt.tight_layout()
plt.show()

## 3 — Run object detection

### 3.1 Show a sample image and run inference on it

Now let's use the **pretrained** `model` from Section 2 (trained on full COCO, so it already recognizes these objects) and run it on one of our sample images.

In [ ]:
sample_image = image_files[0]

results = model.predict(source=str(sample_image), conf=0.25, verbose=False)
result = results[0]  # one Results object per input image

# .plot() draws the predicted boxes + labels + confidence scores on the image
annotated = result.plot()  # returns a BGR numpy array (OpenCV convention)

plt.figure(figsize=(6, 6))
plt.imshow(annotated[..., ::-1])  # convert BGR -> RGB for correct colors
plt.axis("off")
plt.title("Predictions")
plt.show()

### 3.2 Inspect the prediction output

The `result.boxes` object holds everything the model predicted for this image:

- `xyxy`: box corners `[x1, y1, x2, y2]` in pixels
- `conf`: the confidence score for each box
- `cls`: the predicted class id for each box

Let's print these raw values.

In [ ]:
print(f"Number of detected objects: {len(result.boxes)}\n")

for i, box in enumerate(result.boxes):
    xyxy = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
    conf = box.conf.item()
    cls_id = int(box.cls.item())
    print(
        f"Box {i}: class={class_names[cls_id]:<12} "
        f"confidence={conf:.2f}  box=({xyxy[0]:.0f}, {xyxy[1]:.0f}, {xyxy[2]:.0f}, {xyxy[3]:.0f})"
    )

### 3.3 Confidence score

**Definition:** The confidence score (0 to 1) is the model's own estimate of how sure it is that a predicted box (a) contains an object at all, and (b) is correctly classified.

**Conceptual example:** If the model predicts a box labeled `"dog"` with confidence `0.92`, it means the model is 92% sure that box actually contains a dog. A box with confidence `0.10` is likely noise/background and is usually discarded.

In code, the `conf=0.25` argument we used above means: *"only keep predictions the model is at least 25% confident about."* Raising this threshold gives fewer, more reliable boxes; lowering it gives more boxes but more false alarms.

### 3.4 Intersection over Union (IoU)

**Definition:** IoU measures how much two boxes overlap. It is the area where the boxes intersect, divided by the total area they cover together:

$$IoU = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

IoU is always between 0 (no overlap) and 1 (perfect overlap).

**Conceptual example:** Imagine a ground-truth box around a cat, and a predicted box that covers roughly the same region but is shifted slightly to the right.
- If the boxes overlap almost completely → IoU close to `1.0` (great prediction).
- If the boxes only partially overlap → IoU around `0.5` (rough prediction).
- If the boxes barely touch → IoU close to `0.0` (bad prediction — likely a different object, or a miss).

Let's compute IoU for two example boxes with code.

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes, each given as [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0


# Ground-truth box vs. a slightly shifted "predicted" box
ground_truth_box = [50, 50, 150, 150]
predicted_box_good = [60, 60, 160, 160]   # good overlap
predicted_box_bad = [140, 140, 240, 240]  # barely touching

print(f"IoU (good prediction): {compute_iou(ground_truth_box, predicted_box_good):.2f}")
print(f"IoU (bad prediction):  {compute_iou(ground_truth_box, predicted_box_bad):.2f}")

# Visualize the two cases
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, pred_box, title in zip(
    axes, [predicted_box_good, predicted_box_bad], ["Good overlap", "Poor overlap"]
):
    ax.add_patch(patches.Rectangle((ground_truth_box[0], ground_truth_box[1]),
                                    ground_truth_box[2] - ground_truth_box[0],
                                    ground_truth_box[3] - ground_truth_box[1],
                                    linewidth=2, edgecolor="lime", facecolor="none", label="ground truth"))
    ax.add_patch(patches.Rectangle((pred_box[0], pred_box[1]),
                                    pred_box[2] - pred_box[0], pred_box[3] - pred_box[1],
                                    linewidth=2, edgecolor="red", facecolor="none", label="prediction"))
    ax.set_xlim(0, 260)
    ax.set_ylim(260, 0)
    ax.set_title(f"{title}\nIoU = {compute_iou(ground_truth_box, pred_box):.2f}")
    ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

### 3.5 Evaluation IoU vs. NMS IoU

IoU shows up in **two different places** in an object detection pipeline, and it's easy to confuse them:

| | **Evaluation IoU threshold** | **NMS IoU threshold** |
|---|---|---|
| **Used when** | Comparing a prediction to the *ground truth* to decide if it's correct | Comparing predictions to *each other*, before evaluation, to remove duplicates |
| **Purpose** | Defines "correct enough to count as a match" for computing precision/recall/mAP | Prevents the same object being reported multiple times |
| **Example** | mAP50 = mean Average Precision using an evaluation IoU threshold of 0.5 | `model.predict(iou=0.7)` merges overlapping boxes for the *same* object |

In short: **evaluation IoU** grades the model against the ground truth; **NMS IoU** cleans up the model's own raw output before we look at it.

### 3.6 Non-Maximum Suppression (NMS)

**Definition:** Detectors often propose several overlapping boxes for the *same* object. NMS keeps only the best one and removes ("suppresses") the rest by:
1. Keeping the box with the highest confidence score.
2. Removing any other box that overlaps it (IoU) above a chosen **NMS IoU threshold**.
3. Repeating for the remaining boxes.

**Conceptual example:** Suppose the model proposes 3 boxes around the same dog with confidences `0.95`, `0.88`, `0.40`, and the first two overlap heavily (IoU = 0.9) with each other:
- NMS keeps the `0.95` box (highest confidence).
- The `0.88` box is suppressed because it overlaps the `0.95` box above the NMS IoU threshold (e.g. 0.7).
- The `0.40` box, if it doesn't overlap much, is kept as a separate detection.

Result: instead of 3 overlapping boxes for one dog, we get 1 clean box. Let's see this in action by changing the NMS IoU threshold used during inference.

In [ ]:
# Pick an image likely to have several/overlapping objects
crowded_image = image_files[5]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, nms_iou in zip(axes, [0.3, 0.7, 0.95]):
    r = model.predict(source=str(crowded_image), conf=0.25, iou=nms_iou, verbose=False)[0]
    annotated = r.plot()
    ax.imshow(annotated[..., ::-1])
    ax.set_title(f"NMS IoU = {nms_iou}\n{len(r.boxes)} boxes kept")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Lower NMS IoU threshold -> more aggressive suppression -> fewer, cleaner boxes
# Higher NMS IoU threshold -> less suppression -> more (possibly duplicate) boxes kept

## 4 — Train and evaluate the model

### 4.1 Set the training parameters

We'll fine-tune YOLO11n on COCO128 for a small number of epochs — enough to see the training process end-to-end without waiting too long. Feel free to increase `epochs` later for better results.

- `data`: path/name of the dataset config file (`coco128.yaml`)
- `epochs`: number of full passes over the training data
- `imgsz`: image size (pixels) the model is trained on
- `batch`: number of images per training step

In [ ]:
train_params = dict(
    data="coco128.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    project="runs",   # where results get saved
    name="yolo11n_coco128",
)
train_params

### 4.2 Train YOLO11n

We start from a **fresh** (not pretrained) YOLO11n architecture here so that training on COCO128 actually teaches the model something measurable. (Using the already-pretrained-on-COCO checkpoint would start near-perfect since COCO128 is a subset of COCO.)

In [ ]:
# Build a YOLO11n model from its architecture definition (random weights, not pretrained)
train_model = YOLO("yolo11n.yaml")

train_results = train_model.train(**train_params)

### 4.3 Understand the training output

Each epoch prints a row of numbers. The important ones:

- **`box_loss`**: how wrong the predicted box coordinates are (lower is better)
- **`cls_loss`**: how wrong the predicted class is (lower is better)
- **`dfl_loss`**: a loss that helps refine box boundaries (lower is better)
- **`Precision` / `Recall`**: computed on the validation set after each epoch (see Section 5 for definitions)
- **`mAP50`**: mean Average Precision at an evaluation IoU threshold of 0.5
- **`mAP50-95`**: mean Average Precision averaged over IoU thresholds from 0.5 to 0.95 (a stricter, more thorough metric)

You should see the loss values generally decrease and mAP values generally increase as training progresses.

### 4.4 Training and validation graphs

Ultralytics automatically saves a `results.png` plot with curves for all losses and metrics over training. Let's display it.

In [ ]:
train_dir = train_model.trainer.save_dir  # folder where this run's results were saved
print(f"Training results saved to: {train_dir}")

results_png = Image.open(train_dir / "results.png")
plt.figure(figsize=(14, 8))
plt.imshow(results_png)
plt.axis("off")
plt.show()

## 5 — Evaluation metrics

### 5.1 Load the best trained model

During training, Ultralytics saves the checkpoint with the best validation performance as `best.pt`. Let's load it for a clean, dedicated evaluation.

In [ ]:
best_model_path = train_dir / "weights" / "best.pt"
best_model = YOLO(best_model_path)
print(f"Loaded best model from: {best_model_path}")

### 5.2 Validate the model

`model.val()` runs the model on the validation split and computes all the standard detection metrics for us.

In [ ]:
metrics = best_model.val(data="coco128.yaml")

print(f"\nPrecision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")

### 5.3 True Positive, False Positive, and False Negative

To score detections, every prediction is matched against the ground truth using the **evaluation IoU threshold** from Section 3.5 (e.g. 0.5):

- **True Positive (TP):** predicted box matches a real object (IoU ≥ threshold, correct class) ✅
- **False Positive (FP):** predicted box does **not** match any real object — a "hallucinated" detection ❌
- **False Negative (FN):** a real object that the model **failed to detect** ❌

**Conceptual example:** An image has 3 real cats. The model predicts 4 boxes labeled "cat":
- 2 predictions correctly overlap 2 of the real cats → **2 TP**
- 1 prediction lands on empty background → **1 FP**
- The 3rd real cat was never detected → **1 FN**

### 5.4 Precision and recall

$$Precision = \frac{TP}{TP + FP} \qquad Recall = \frac{TP}{TP + FN}$$

- **Precision** answers: *"Of all the boxes I predicted, how many were actually correct?"*
- **Recall** answers: *"Of all the real objects, how many did I successfully find?"*

**Conceptual example (continuing above):** TP=2, FP=1, FN=1

$$Precision = \frac{2}{2+1} = 0.67 \qquad Recall = \frac{2}{2+1} = 0.67$$

A model that predicts very few boxes (only when extremely sure) tends to have **high precision, low recall**. A model that predicts many boxes tends to have **high recall, low precision**.

### 5.5 F1 score

The F1 score is the **harmonic mean** of precision and recall — a single number that balances both:

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

**Conceptual example:** with Precision = 0.67 and Recall = 0.67:

$$F1 = 2 \times \frac{0.67 \times 0.67}{0.67 + 0.67} = 0.67$$

F1 is useful because a model can't get a high F1 just by being extreme in one direction (e.g. predicting everything to maximize recall) — both precision and recall need to be reasonably good.

### 5.6 Confusion matrix and PR curve

- The **confusion matrix** shows, for every class, how many predictions were correct vs. confused with another class vs. missed/background.
- The **Precision-Recall (PR) curve** shows the precision/recall trade-off as the confidence threshold changes, and its area is the **Average Precision (AP)** for that class.

Ultralytics saves both automatically during `val()`. Let's display them.

In [ ]:
val_dir = metrics.save_dir  # folder where val() saved its plots

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(Image.open(val_dir / "confusion_matrix_normalized.png"))
axes[0].set_title("Confusion Matrix (normalized)")
axes[0].axis("off")

axes[1].imshow(Image.open(val_dir / "PR_curve.png"))
axes[1].set_title("Precision-Recall Curve")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 6 — Parameter experiments

### 6.1 Confidence threshold experiment

Recall from Section 3.3: the confidence threshold (`conf`) controls how sure the model must be before it reports a detection. Let's run inference on the same image with several thresholds and see how the number of kept boxes changes.

In [ ]:
conf_thresholds = [0.1, 0.25, 0.5, 0.75]

fig, axes = plt.subplots(1, len(conf_thresholds), figsize=(20, 5))
for ax, conf in zip(axes, conf_thresholds):
    r = best_model.predict(source=str(crowded_image), conf=conf, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1])
    ax.set_title(f"conf = {conf}\n{len(r.boxes)} boxes kept")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Lower conf -> more (less certain) boxes kept -> higher recall, lower precision
# Higher conf -> fewer (more certain) boxes kept -> higher precision, lower recall

### 6.2 NMS IoU threshold experiment

Recall from Section 3.6: the NMS IoU threshold (`iou`) controls how aggressively overlapping boxes are merged. Let's repeat the experiment, this time varying `iou` while keeping `conf` fixed.

In [ ]:
nms_iou_thresholds = [0.3, 0.5, 0.7, 0.9]

fig, axes = plt.subplots(1, len(nms_iou_thresholds), figsize=(20, 5))
for ax, nms_iou in zip(axes, nms_iou_thresholds):
    r = best_model.predict(source=str(crowded_image), conf=0.25, iou=nms_iou, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1])
    ax.set_title(f"NMS IoU = {nms_iou}\n{len(r.boxes)} boxes kept")
    ax.axis("off")
plt.tight_layout()
plt.show()

# Lower NMS IoU threshold -> aggressive suppression -> risk of merging distinct nearby objects
# Higher NMS IoU threshold -> lenient suppression -> risk of duplicate boxes on the same object